In [1]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [2]:
from protest_impact.data.protests.detection import load_aglpn_dataset, load_glpn_dataset

glpn = load_glpn_dataset()
aglpn = load_aglpn_dataset()

Using custom data configuration default-386fb51d30c78e56
Found cached dataset csv (/Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317)


  0%|          | 0/5 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-1c6c61baf1fc4966.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-bbadab134ac9ce97.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-5b646fe4fc3137d1.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317/cache-52989505097dfa79.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/csv/default-386fb51d30c78e56/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317

  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-f5be8468b0beb351.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-305a1cf381866fe8.arrow
Loading cached split indices for dataset at /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-bcff46202590e9cd.arrow and /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-286ec6731b65d5bd.arrow
Using custom data configuration protest-183a20b608d37f99
Found cached dataset json (/Users/david/.cache/huggingface/datasets/json/protest-183a20b608d37f99/0.0.0/0f7e3662623656454fcd2b650f34e886a7d

  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-183a20b608d37f99/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-aa5ef0423c656460.arrow


In [3]:
import optuna
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from datasets import concatenate_datasets
from protest_impact.data.news import kwic_dataset
from protest_impact.data.protests.detection import load_aglpn_dataset, load_glpn_dataset
from protest_impact.data.protests.detection.train import evaluate_, train_model


def objective(trial, return_model=False):
    dataset_name = trial.suggest_categorical(
        "dataset",
        [
            "aglpn",
            "aglpn+glpn",
            "aglpn+aglpn.positive",
            "aglpn+aglpn.positive+glpn",
        ],
    )
    if dataset_name == "aglpn":
        train = load_aglpn_dataset()["train"]
    elif dataset_name == "aglpn+glpn":
        train = concatenate_datasets(
            [load_aglpn_dataset()["train"], load_glpn_dataset()["train"]]
        )
    elif dataset_name == "aglpn+aglpn.positive":
        train = concatenate_datasets(
            [load_aglpn_dataset()["train"], load_aglpn_dataset()["train.positive"]]
        )
    elif dataset_name == "aglpn+aglpn.positive+glpn":
        train = concatenate_datasets(
            [
                load_aglpn_dataset()["train"],
                load_aglpn_dataset()["train.positive"],
                load_glpn_dataset()["train"],
            ]
        )
    n_kwic = trial.suggest_int("n_kwic", 1, 5)
    train = kwic_dataset(train, n=n_kwic)
    test = load_aglpn_dataset()["test"]
    test = kwic_dataset(test, n=n_kwic)
    model_name = "deepset/gelectra-large"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_vanilla = AutoModelForSequenceClassification.from_pretrained(model_name).to(
        device
    )
    model = train_model(model_vanilla, tokenizer, dataset_name, train)
    if return_model:
        return model
    evaluation = evaluate_(model, tokenizer, test)
    return evaluation["f1"]


study = optuna.create_study(
    direction="maximize",
    study_name="protest_detection_v0",
    load_if_exists=True,
    storage="sqlite:///db.sqlite3",
)
study.optimize(objective, n_trials=2)

[I 2023-02-12 14:03:39,841] Using an existing study with name 'protest_detection_v0' instead of creating a new one.
Using custom data configuration protest-a60c8e2a0ed20541
Found cached dataset json (/Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-f5be8468b0beb351.arrow
Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-305a1cf381866fe8.arrow
Loading cached split indices for dataset at /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-bcff46202590e9cd.arrow and /Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-286ec6731b65d5bd.arrow
Using custom data configuration protest-183a20b608d37f99
Found cached dataset json (/Users/david/.cache/huggingface/datasets/json/protest-183a20b608d37f99/0.0.0/0f7e3662623656454fcd2b650f34e886a7d

  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at /Users/david/.cache/huggingface/datasets/json/protest-183a20b608d37f99/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51/cache-aa5ef0423c656460.arrow
Using custom data configuration protest-a60c8e2a0ed20541
Found cached dataset json (/Users/david/.cache/huggingface/datasets/json/protest-a60c8e2a0ed20541/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51)


...

  0%|          | 0/500 [00:00<?, ?ex/s]

Some weights of the model checkpoint at deepset/gelectra-large were not used when initializing ElectraForSequenceClassification: ['discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.weight']
- This IS expected if you are initializing ElectraForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at deepset/gelectra-large and are newly initialized: ['classifier.dense.bias', 'classifie

  0%|          | 0/2 [00:00<?, ?ba/s]

[W 2023-02-12 14:05:10,062] Trial 3 failed with parameters: {'dataset': 'aglpn+aglpn.positive', 'n_kwic': 1} because of the following error: ValueError('FP16 Mixed precision training with AMP or APEX (`--fp16`) and FP16 half precision evaluation (`--fp16_full_eval`) can only be used on CUDA devices.').
Traceback (most recent call last):
  File "/Users/david/Repositories/protest-impact/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/6v/w9nn6c_n4qdbrjwfnq7695n00000gn/T/ipykernel_21406/1711848213.py", line 45, in objective
    model = train_model(model_vanilla, tokenizer, dataset_name, train)
  File "/Users/david/Repositories/protest-impact/protest_impact/data/protests/detection/train.py", line 33, in train_model
    training_args = TrainingArguments(
  File "<string>", line 108, in __init__
  File "/Users/david/Repositories/protest-impact/.venv/lib/python3.10/site-packages/transformers/training_

ValueError: FP16 Mixed precision training with AMP or APEX (`--fp16`) and FP16 half precision evaluation (`--fp16_full_eval`) can only be used on CUDA devices.